# PyTorch Fundamentals — Exercise Solutions (Amr)

<a href="https://colab.research.google.com/github/amr-elgharip2006/dl-pytorch-fundamentals-exercises/blob/main/DL_00_Exercises_Solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Solutions only — a short view of the exercises at the end of `00_pytorch_fundamentals`. The full section notebook with these solutions in place is [`DL_00_Exercises_Amr.ipynb`](DL_00_Exercises_Amr.ipynb).

## Exercise Solutions — Amr

Solutions to the 10 exercises above and the challenge. Each exercise has its own header, then the code and a short explanation. These cells only need the setup cell below, so they can also be run on their own.

> **Environment note:** I ran this notebook on an Apple Silicon Mac. Its accelerator is **MPS** (Apple's Metal backend), so the outputs show tensors with `device='mps:0'`. The code is **device-agnostic**: on Google Colab with a GPU runtime the same cells show `device='cuda:0'`, and on a CPU-only runtime everything stays on `cpu`. Random numbers generated **directly on** an accelerator (Exercise 5) also differ between backends. Random numbers generated on the CPU and then moved with `.to(device)` (Exercises 2–4, 6–10) are the same on every machine.

In [1]:
# Setup for the solutions: import PyTorch and pick the device in a device-agnostic way
import torch

print(f"PyTorch version: {torch.__version__}")

# Same pattern as earlier in the notebook: use the accelerator if one is available, otherwise the CPU
device = (
    torch.accelerator.current_accelerator()
    if torch.accelerator.is_available()
    else torch.device("cpu")
)

# Equivalent one-liner that gives the device as a plain string ("cuda", "mps", "xpu", ... or "cpu")
device_str = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

print(f"Selected device: {device}  (string form: '{device_str}')")
print(f"Both forms point to the same device? {torch.device(device_str) == device}")

PyTorch version: 2.12.0
Selected device: mps  (string form: 'mps')
Both forms point to the same device? True


### Exercise 1 — Key takeaways from the `torch.Tensor` and `torch.accelerator` docs

**`torch.Tensor`**

- A `torch.Tensor` is a multi-dimensional array where **all elements share one data type**. Check three attributes first: `dtype` (floating-point tensors default to `torch.float32`), `shape` (the same as `.size()`), and `device` (`cpu`, `cuda:0`, `mps:0`, ...). Most PyTorch errors come from a mismatch in one of them.
- **Common creation ops:** `torch.tensor(data)` (copies the data), `torch.zeros` / `torch.ones` / `torch.full`, `torch.rand` (uniform in `[0, 1)`) and `torch.randn` (standard normal), `torch.arange`, and the `*_like` versions (e.g. `torch.zeros_like(x)`), which reuse another tensor's shape, dtype and device. `torch.from_numpy` creates a tensor from a NumPy array. Most of these accept `dtype=` and `device=` arguments.
- **In-place methods end with `_`** (`add_()`, `zero_()`, `clamp_()`, ...): they change the tensor itself. The version without `_` (`add()`) returns a new tensor and leaves the original unchanged.
- `.to(...)` changes dtype and/or device and **returns** a tensor, so you have to reassign the result. `.item()` converts a one-element tensor to a Python number, and `.numpy()` only works on CPU tensors.

**`torch.accelerator`**

- `torch.accelerator` is PyTorch's **device-agnostic API** for "the current accelerator", whichever backend that is: **CUDA, MPS, XPU, MTIA** or another one. The same code works on all of them without hard-coding `"cuda"`.
- `torch.accelerator.is_available()` returns `True` only if an accelerator backend is built in, its drivers are present, and at least one device is visible at runtime.
- `torch.accelerator.current_accelerator()` returns a `torch.device` such as `device(type='cuda')`. It has no index; `current_device_index()` gives the index. Its docstring says it reports the accelerator available at **compile time**, so a CUDA build can return `cuda` even on a CPU-only runtime. That is why it is guarded with `is_available()` (or called with `check_available=True`).
- `torch.accelerator.device_count()` returns the number of devices (0 if there are none). `torch.accelerator.synchronize()` **waits until all queued work on the device has finished**. Accelerator operations run asynchronously, so call it before timing GPU code. The module also has `set_device_index()` plus stream and memory helpers, but backend support for the memory helpers varies.

In [2]:
# Exercise 1 — quick checks of what the docs describe

# torch.Tensor: the three key attributes
example = torch.tensor([[1.0, 2.0, 3.0],
                        [4.0, 5.0, 6.0]])
print(f"dtype: {example.dtype} | shape: {example.shape} | device: {example.device}")

# Out-of-place vs in-place (_)
plus_ten = example.add(10)   # returns a NEW tensor, `example` is unchanged
example.add_(10)             # trailing underscore -> modifies `example` itself
print(f"`example` after add_(10):\n{example}")
print(f"Same values as the out-of-place result? {torch.equal(example, plus_ten)}\n")

# torch.accelerator: confirm the functions used in this notebook exist in the installed version
for name in ["is_available", "current_accelerator", "device_count", "current_device_index", "synchronize"]:
    print(f"torch.accelerator.{name:<21} exists: {hasattr(torch.accelerator, name)}")

print(f"\nis_available():                            {torch.accelerator.is_available()}")
print(f"current_accelerator(check_available=True): {torch.accelerator.current_accelerator(check_available=True)}")
print(f"device_count():                            {torch.accelerator.device_count()}")
if torch.accelerator.is_available():
    print(f"current_device_index():                    {torch.accelerator.current_device_index()}")
    torch.accelerator.synchronize()  # block until every queued kernel on the accelerator has finished
    print("synchronize():                             done")

dtype: torch.float32 | shape: torch.Size([2, 3]) | device: cpu
`example` after add_(10):
tensor([[11., 12., 13.],
        [14., 15., 16.]])
Same values as the out-of-place result? True

torch.accelerator.is_available          exists: True
torch.accelerator.current_accelerator   exists: True
torch.accelerator.device_count          exists: True
torch.accelerator.current_device_index  exists: True
torch.accelerator.synchronize           exists: True

is_available():                            True
current_accelerator(check_available=True): mps
device_count():                            1
current_device_index():                    0
synchronize():                             done


### Exercise 2 — Random tensor with shape `(7, 7)`

`torch.rand` samples values uniformly from `[0, 1)`. There is no seed yet, so the values change each time the cell runs (Exercise 4 adds a seed).

In [3]:
# Create a random tensor of shape (7, 7)
tensor_7x7 = torch.rand(size=(7, 7))

print(tensor_7x7)
print(f"Shape: {tensor_7x7.shape}")

tensor([[0.3432, 0.0460, 0.2048, 0.7626, 0.5486, 0.7150, 0.6685],
        [0.9591, 0.8216, 0.9018, 0.3504, 0.1360, 0.6601, 0.1868],
        [0.6095, 0.8745, 0.1995, 0.3813, 0.5108, 0.6394, 0.7559],
        [0.4015, 0.6618, 0.5429, 0.4301, 0.1701, 0.5655, 0.0895],
        [0.6631, 0.3882, 0.1564, 0.0672, 0.2143, 0.6802, 0.0882],
        [0.3895, 0.9376, 0.6129, 0.8537, 0.9681, 0.9783, 0.4940],
        [0.4310, 0.8667, 0.1916, 0.4004, 0.2674, 0.6018, 0.2743]])
Shape: torch.Size([7, 7])


### Exercise 3 — Matrix multiplication with a second random tensor

**Inner-dimension rule:** for `A @ B`, the number of **columns of `A`** must equal the number of **rows of `B`**:

$$(m, \mathbf{n}) \;@\; (\mathbf{n}, p) \;\rightarrow\; (m, p)$$

The result takes the **outer** dimensions. With a second tensor of shape `(1, 7)`, `(7, 7) @ (1, 7)` fails because the inner dimensions are `7` and `1`. Transposing it to `(7, 1)` makes them `7` and `7`, so `(7, 7) @ (7, 1) → (7, 1)`.

In [4]:
# Second random tensor with shape (1, 7)
tensor_1x7 = torch.rand(size=(1, 7))

# Without a transpose the inner dimensions don't match (7 vs 1) -> error
try:
    torch.matmul(tensor_7x7, tensor_1x7)
except RuntimeError as error:
    print(f"Without transpose -> RuntimeError: {error}\n")

# Transpose (1, 7) -> (7, 1) so the inner dimensions match (7 and 7)
result_7x1 = torch.matmul(tensor_7x7, tensor_1x7.T)

print(f"{tuple(tensor_7x7.shape)} @ {tuple(tensor_1x7.T.shape)} -> {tuple(result_7x1.shape)}\n")
print(result_7x1)

Without transpose -> RuntimeError: mat1 and mat2 shapes cannot be multiplied (7x7 and 1x7)

(7, 7) @ (7, 1) -> (7, 1)

tensor([[2.0236],
        [1.6569],
        [1.8812],
        [1.3449],
        [1.1593],
        [2.8756],
        [1.4371]])


### Exercise 4 — Set the seed to `0` and repeat Exercises 2–3

To compare runs, Exercises 2–3 are wrapped in a function. Each call can reset the seed first, so two seeded runs can be checked with `torch.equal`.

In [5]:
def exercises_2_and_3(seed=None):
    # Create a (7, 7) and a (1, 7) random tensor, then multiply them (after transposing)
    if seed is not None:
        torch.manual_seed(seed)          # reset the random number generator to a known state
    X = torch.rand(size=(7, 7))          # Exercise 2
    Y = torch.rand(size=(1, 7))          # second tensor for Exercise 3
    return X, Y, X @ Y.T                 # (7, 7) @ (7, 1) -> (7, 1)

# Two runs, both starting from seed 0
X1, Y1, Z1 = exercises_2_and_3(seed=0)
X2, Y2, Z2 = exercises_2_and_3(seed=0)

print(f"Random (7, 7) tensor with seed 0:\n{X1}\n")
print(f"Result with seed 0, shape {tuple(Z1.shape)}:\n{Z1}\n")
print(f"Run 1 vs Run 2 -> X equal? {torch.equal(X1, X2)} | Y equal? {torch.equal(Y1, Y2)} | result equal? {torch.equal(Z1, Z2)}")

# A third run WITHOUT re-seeding continues from the generator's current state -> different numbers
_, _, Z3 = exercises_2_and_3(seed=None)
print(f"Run 3 (no re-seed) vs Run 1 -> result equal? {torch.equal(Z1, Z3)}")

Random (7, 7) tensor with seed 0:
tensor([[0.4963, 0.7682, 0.0885, 0.1320, 0.3074, 0.6341, 0.4901],
        [0.8964, 0.4556, 0.6323, 0.3489, 0.4017, 0.0223, 0.1689],
        [0.2939, 0.5185, 0.6977, 0.8000, 0.1610, 0.2823, 0.6816],
        [0.9152, 0.3971, 0.8742, 0.4194, 0.5529, 0.9527, 0.0362],
        [0.1852, 0.3734, 0.3051, 0.9320, 0.1759, 0.2698, 0.1507],
        [0.0317, 0.2081, 0.9298, 0.7231, 0.7423, 0.5263, 0.2437],
        [0.5846, 0.0332, 0.1387, 0.2422, 0.8155, 0.7932, 0.2783]])

Result with seed 0, shape (7, 1):
tensor([[1.8542],
        [1.9611],
        [2.2884],
        [3.0481],
        [1.7067],
        [2.5290],
        [1.7989]])

Run 1 vs Run 2 -> X equal? True | Y equal? True | result equal? True
Run 3 (no re-seed) vs Run 1 -> result equal? False


**Why this works:** `torch.manual_seed(0)` resets the generator to a fixed state, and every random call after that moves the state forward.

- To reproduce a random tensor (or a sequence of them), **set the seed right before the random call(s) you want to repeat**. That is why the function seeds at the start of every run.
- Within one run, `X` and `Y` are still different from each other because the generator moved on between the two calls. The **pair** is identical every time.
- Run 3 skipped the seed, so it continued from the current state and gave new numbers.

### Exercise 5 — How `torch.manual_seed()` relates to accelerator RNGs

PyTorch keeps a **separate random number generator (RNG) per device**: one for the CPU, and one for each accelerator device (every CUDA GPU has its own; MPS has one).

- **`torch.manual_seed(seed)` seeds them all.** In the installed PyTorch (2.12), its docstring says it sets the seed "for generating random numbers on all devices". Its source shows that it seeds the CPU default generator **and** calls `torch.cuda.manual_seed_all`, `torch.mps.manual_seed`, `torch.xpu.manual_seed_all` and `torch.mtia.manual_seed_all` (plus any custom backend). It is safe to call on a machine without a GPU, and it returns the CPU generator `torch.default_generator`.
- **Backend-specific functions seed only that backend** and leave the CPU generator alone. `torch.cuda.manual_seed(seed)` seeds only the **current** GPU, and `torch.cuda.manual_seed_all(seed)` seeds **all** GPUs; both silently do nothing if CUDA isn't available. `torch.mps.manual_seed(seed)` does the same for Apple MPS.
- **The same seed does NOT give the same numbers on CPU and GPU.** `torch.rand(3)` and `torch.rand(3, device="cuda")` use different generators with different algorithms (a Mersenne Twister on the CPU, Philox on CUDA). The values differ, and they can also differ between backends (CUDA vs MPS) and between PyTorch versions. To get identical data everywhere, **generate on the CPU, then `.to(device)`**.
- **A seed makes random numbers repeatable, but not every computation.** Some GPU kernels (some cuDNN convolutions, atomic-add based scatter/index ops, ...) are non-deterministic. Full determinism also needs `torch.use_deterministic_algorithms(True)` (on CUDA this also requires the environment variable `CUBLAS_WORKSPACE_CONFIG=:4096:8`), `torch.backends.cudnn.deterministic = True` and `torch.backends.cudnn.benchmark = False`, plus seeding Python's `random` and NumPy if you use them.

The code below demonstrates these points on whichever device is available.

In [6]:
# Exercise 5 — demonstrating how the CPU and accelerator generators are seeded
print(f"Using device: {device}\n")

# 1) torch.manual_seed() returns the CPU default generator
gen = torch.manual_seed(42)
print(f"1) manual_seed returned the CPU default generator? {gen is torch.default_generator} "
      f"(torch.initial_seed() = {torch.initial_seed()})\n")

# 2) Same seed -> same numbers ON THE ACCELERATOR too (its generator was also seeded)
torch.manual_seed(42)
acc_run1 = torch.rand(3, device=device)
torch.manual_seed(42)
acc_run2 = torch.rand(3, device=device)
print(f"2) {device.type} run 1: {acc_run1}")
print(f"   {device.type} run 2: {acc_run2}")
print(f"   Identical? {torch.equal(acc_run1, acc_run2)}\n")

# 3) Same seed, CPU vs accelerator -> different generators, so (generally) different numbers
torch.manual_seed(42)
cpu_values = torch.rand(3)
print(f"3) cpu   with seed 42: {cpu_values}")
print(f"   {device.type:<5} with seed 42: {acc_run1.cpu()}")
if device.type == "cpu":
    print("   No accelerator here, so both used the CPU generator and they match.")
else:
    print(f"   CPU values == {device.type} values? {torch.equal(cpu_values, acc_run1.cpu())}\n")

    # 4) Seeding ONLY the CPU generator does not reset the accelerator generator
    #    (step 3's torch.manual_seed(42) also re-seeded the accelerator, so first draw once
    #     on the accelerator to move its generator past the "run 1" numbers)
    _ = torch.rand(3, device=device)
    torch.default_generator.manual_seed(42)      # re-seed the CPU generator only
    after_cpu_only_seed = torch.rand(3, device=device)
    print(f"4) After seeding only the CPU generator, does {device.type} repeat run 1? "
          f"{torch.equal(after_cpu_only_seed, acc_run1)}")

    # 5) Seeding ONLY the accelerator with its backend function reproduces run 1
    backend = torch.get_device_module(device)    # torch.cuda, torch.mps, torch.xpu, ...
    if hasattr(backend, "manual_seed_all"):
        backend.manual_seed_all(42)              # CUDA / XPU: seed every GPU
        seed_fn = f"{backend.__name__}.manual_seed_all"
    else:
        backend.manual_seed(42)                  # MPS: a single device
        seed_fn = f"{backend.__name__}.manual_seed"
    after_backend_seed = torch.rand(3, device=device)
    print(f"5) After {seed_fn}(42), does {device.type} repeat run 1? "
          f"{torch.equal(after_backend_seed, acc_run1)}")

Using device: mps

1) manual_seed returned the CPU default generator? True (torch.initial_seed() = 42)

2) mps run 1: tensor([0.8172, 0.5881, 0.5581], device='mps:0')
   mps run 2: tensor([0.8172, 0.5881, 0.5581], device='mps:0')
   Identical? True

3) cpu   with seed 42: tensor([0.8823, 0.9150, 0.3829])
   mps   with seed 42: tensor([0.8172, 0.5881, 0.5581])
   CPU values == mps values? False

4) After seeding only the CPU generator, does mps repeat run 1? False
5) After torch.mps.manual_seed(42), does mps repeat run 1? True


**What the output shows:** `torch.manual_seed(42)` made the accelerator results repeatable (2), but the CPU and accelerator still produced different numbers from the same seed (3). Re-seeding only the CPU generator did **not** reset the accelerator generator (4). The backend's own seeding function did (5). So `torch.manual_seed` works by seeding every device's generator, not by sharing one generator.

A typical "make everything reproducible" helper for training scripts (not run here, because deterministic mode can make some CUDA ops raise errors unless `CUBLAS_WORKSPACE_CONFIG` is set before CUDA starts):

```python
import random, numpy as np

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)                     # CPU + all accelerator devices
    torch.backends.cudnn.deterministic = True   # only affects CUDA/cuDNN
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)    # error on ops with no deterministic version
```

### Exercise 6 — Two random `(2, 3)` tensors on the selected `device`

`device` was selected in the setup cell with the device-agnostic pattern: `torch.accelerator.current_accelerator()` if an accelerator is available, else `"cpu"`. The same code therefore uses CUDA on Colab, MPS on a Mac, or the CPU.

The seed is `1234`, the value used in the original version of this exercise; any fixed seed works. The tensors are created on the CPU and then moved with `.to(device)`, so the values are the same on every machine (see Exercise 5).

In [7]:
# `device` comes from the setup cell (accelerator if available, otherwise CPU)
print(f"Device: {device}\n")

# Set the seed (1234 as in the original exercise; any fixed seed works)
torch.manual_seed(1234)

# Create two random (2, 3) tensors on the CPU, then move them to the target device
tensor_A = torch.rand(size=(2, 3)).to(device)
tensor_B = torch.rand(size=(2, 3)).to(device)

print(f"tensor_A:\n{tensor_A}\n")
print(f"tensor_B:\n{tensor_B}")

Device: mps



tensor_A:
tensor([[0.0290, 0.4019, 0.2598],
        [0.3666, 0.0583, 0.7006]], device='mps:0')

tensor_B:
tensor([[0.0518, 0.4681, 0.6738],
        [0.3315, 0.7837, 0.5631]], device='mps:0')


### Exercise 7 — Matrix multiplication after adjusting the shapes

`(2, 3) @ (2, 3)` fails because the inner dimensions are `3` and `2`. Transposing `tensor_B` gives `(2, 3) @ (3, 2) → (2, 2)`. Both tensors are on the same device, so the multiplication runs on that device.

In [8]:
# Transpose tensor_B: (2, 3) -> (3, 2), so the inner dimensions match (3 and 3)
tensor_C = torch.matmul(tensor_A, tensor_B.T)

print(f"{tuple(tensor_A.shape)} @ {tuple(tensor_B.T.shape)} -> {tuple(tensor_C.shape)}\n")
print(tensor_C)

(2, 3) @ (3, 2) -> (2, 2)



tensor([[0.3647, 0.4709],
        [0.5184, 0.5617]], device='mps:0')


### Exercise 8 — Maximum and minimum of the result

In [9]:
# Reduce the whole (2, 2) tensor to a single max / min value
max_value = tensor_C.max()
min_value = tensor_C.min()

print(f"Max: {max_value!r}")
print(f"Min: {min_value!r}")

# .item() copies the single value back to the CPU as a plain Python float
print(f"As Python numbers -> max = {max_value.item():.4f}, min = {min_value.item():.4f}")

Max: tensor(0.5617, device='mps:0')
Min: tensor(0.3647, device='mps:0')
As Python numbers -> max = 0.5617, min = 0.3647


### Exercise 9 — Indices of the maximum and minimum values

`argmax()` / `argmin()` without a `dim` argument return an index into the **flattened** tensor. `torch.unravel_index` converts that flat index into a `(row, col)` index for the original `(2, 2)` shape.

In [10]:
# Flat indices (positions in tensor_C.flatten())
argmax_flat = tensor_C.argmax()
argmin_flat = tensor_C.argmin()
print(f"Flattened tensor: {tensor_C.flatten()}")
print(f"Index of max (flat): {argmax_flat!r}")
print(f"Index of min (flat): {argmin_flat!r}\n")

# Convert the flat indices into (row, col) indices for the (2, 2) tensor
max_row, max_col = torch.unravel_index(argmax_flat, tensor_C.shape)
min_row, min_col = torch.unravel_index(argmin_flat, tensor_C.shape)
print(f"Max is at (row, col) = ({max_row.item()}, {max_col.item()}) -> {tensor_C[max_row, max_col].item():.4f}")
print(f"Min is at (row, col) = ({min_row.item()}, {min_col.item()}) -> {tensor_C[min_row, min_col].item():.4f}")

Flattened tensor: tensor([0.3647, 0.4709, 0.5184, 0.5617], device='mps:0')
Index of max (flat): tensor(3, device='mps:0')
Index of min (flat): tensor(0, device='mps:0')



Max is at (row, col) = (1, 1) -> 0.5617
Min is at (row, col) = (0, 0) -> 0.3647


### Exercise 10 — Seed `7`, shape `(1, 1, 1, 10)`, then squeeze to `(10)`

`squeeze()` with no arguments removes **every** dimension of size 1, so `(1, 1, 1, 10) → (10,)`. The data is unchanged; only the shape changes.

In [11]:
# Set the seed before creating the tensor so the values are reproducible
torch.manual_seed(7)
tensor_4d = torch.rand(size=(1, 1, 1, 10))

# Remove all singleton (size-1) dimensions
tensor_1d = tensor_4d.squeeze()

print(f"Original tensor:\n{tensor_4d}\nShape: {tensor_4d.shape}\n")
print(f"Squeezed tensor:\n{tensor_1d}\nShape: {tensor_1d.shape}")

Original tensor:
tensor([[[[0.5349, 0.1988, 0.6592, 0.6569, 0.2328, 0.4251, 0.2071, 0.6297,
           0.3653, 0.8513]]]])
Shape: torch.Size([1, 1, 1, 10])

Squeezed tensor:
tensor([0.5349, 0.1988, 0.6592, 0.6569, 0.2328, 0.4251, 0.2071, 0.6297, 0.3653,
        0.8513])
Shape: torch.Size([10])


### Challenge — Tracking `shape`, `dtype` and `device` through Exercises 6–9

The code below repeats Exercises 6–9 and prints the three attributes at every important step. The table after it explains each value.

In [12]:
def describe(step, t):
    # Print the three attributes behind most PyTorch errors
    print(f"{step:<34} shape={str(tuple(t.shape)):<8} dtype={str(t.dtype):<13} device={t.device}")

torch.manual_seed(1234)

# Exercise 6: create on the CPU, then move to the device
a_cpu = torch.rand(size=(2, 3));  describe("1. torch.rand(2, 3)", a_cpu)
b_cpu = torch.rand(size=(2, 3))
a = a_cpu.to(device);             describe("2. a_cpu.to(device)", a)
b = b_cpu.to(device)

# Exercise 7: transpose and multiply
b_T = b.T;                        describe("3. b.T (transpose)", b_T)
c = a @ b_T;                      describe("4. a @ b.T", c)

# Exercise 8: max / min
c_max = c.max();                  describe("5. c.max()", c_max)

# Exercise 9: argmax / unravel_index
c_argmax = c.argmax();            describe("6. c.argmax()", c_argmax)
row, col = torch.unravel_index(c_argmax, c.shape)
describe("7. unravel_index(...) -> row", row)

# Back to the CPU / plain Python
c_cpu = c.cpu();                  describe("8. c.cpu()", c_cpu)
max_py = c_max.item()
print(f"{'9. c.max().item()':<34} type={type(max_py).__name__} (a plain Python number: no shape/dtype/device)")

1. torch.rand(2, 3)                shape=(2, 3)   dtype=torch.float32 device=cpu
2. a_cpu.to(device)                shape=(2, 3)   dtype=torch.float32 device=mps:0
3. b.T (transpose)                 shape=(3, 2)   dtype=torch.float32 device=mps:0
4. a @ b.T                         shape=(2, 2)   dtype=torch.float32 device=mps:0
5. c.max()                         shape=()       dtype=torch.float32 device=mps:0
6. c.argmax()                      shape=()       dtype=torch.int64   device=mps:0
7. unravel_index(...) -> row       shape=()       dtype=torch.int64   device=mps:0
8. c.cpu()                         shape=(2, 2)   dtype=torch.float32 device=cpu
9. c.max().item()                  type=float (a plain Python number: no shape/dtype/device)


**Why each value is what it is** (the device appears as `mps:0` here; on Colab it is `cuda:0` with a GPU, or `cpu`):

| Step | shape | dtype | device | Why |
|---|---|---|---|---|
| 1. `torch.rand(2, 3)` | `(2, 3)` | `float32` | `cpu` | The shape is what we asked for. `torch.rand` returns floating-point numbers in the **default dtype** (`torch.get_default_dtype()` → `float32`). Tensors are created on the **CPU** unless `device=` is passed. |
| 2. `.to(device)` | `(2, 3)` | `float32` | `mps:0` / `cuda:0` | `.to(device)` only changes **where** the data lives. Shape and dtype stay the same. `:0` means the first device of that type (`current_accelerator()` has no index, but a tensor always sits on a concrete device). On a CPU-only machine nothing changes. |
| 3. `b.T` | `(3, 2)` | `float32` | same as `b` | A transpose swaps the two dimensions. It is a **view** of the same memory, so dtype and device can't change. |
| 4. `a @ b.T` | `(2, 2)` | `float32` | same device | `(2, 3) @ (3, 2)` → the **outer** dimensions `(2, 2)`. Both inputs are `float32` on the same device, so the result is too. (Inputs on different devices would raise an error.) |
| 5. `c.max()` | `()` | `float32` | same device | A full reduction returns a **0-dim tensor** (shape `torch.Size([])`, a scalar). It keeps the input's dtype and **stays on the accelerator**; PyTorch doesn't move results back to the CPU by itself. |
| 6. `c.argmax()` | `()` | `int64` | same device | It returns an **index** (into the flattened tensor), not a value. PyTorch uses `int64` (`torch.long`) for indices. It is a 0-dim tensor on the same device. |
| 7. `unravel_index` | `()` | `int64` | same device | Returns a tuple with one `int64` index tensor per dimension (row, col). |
| 8. `c.cpu()` | `(2, 2)` | `float32` | `cpu` | Copies the data back to host memory. This is needed before `.numpy()` or plotting. |
| 9. `.item()` | — | Python `float` | — | Converts a one-element tensor into a normal Python number. On an accelerator this waits for the device and copies the value to the CPU. |